In [1]:
from de_classes.neo4j_handler import Neo4jUtils
from de_classes.spark_factory import SparkFactory
from de_classes.data_processor import *
from pyspark.sql.types import *

In [2]:
URI = "neo4j+s://0cf2316b.databases.neo4j.io"
neo4j_user = "neo4j"
neo4j_password = "PYIUheqgDra8xkF56RARlNovau4Af05TWzJiMwxu0tI"  

In [3]:
driver = Neo4jUtils(URI, neo4j_user, neo4j_password)

In [4]:
spark = SparkFactory("AirQuality-neo4j").get()

25/08/28 18:24:27 WARN Utils: Your hostname, LAPTOP-5G2QRQLJ.localdomain resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/08/28 18:24:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/28 18:24:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/28 18:24:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/08/28 18:24:32 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/08/28 18:24:32 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [6]:
schema = StructType([
    StructField("Item_Code", IntegerType(),True),
    StructField("Station_Code",  IntegerType(),True),
    StructField("Measurement_Date",  TimestampType(),True),
    StructField("Average_Value",  DoubleType(),True),
    StructField("Instrument_Status",  IntegerType(),True),
    StructField("Station_Name",  StringType(),True),
    StructField("Latitude",  DoubleType(),True),
    StructField("Longitude",  DoubleType(),True),
    StructField("Address",  StringType(),True),
    StructField("Item_Name",  StringType(),True),
    StructField("Unit",  StringType(),True),
    StructField("Status",  StringType(),True),
])

df = DataLoading.read_csv(spark, "hdfs://localhost:9000/user/student/processed_data", schema)

In [7]:
driver.create_constraints()

Executed: 
Executed: 
Executed: 
Executed: 
Executed: 


In [8]:
records = [row.asDict() for row in df.collect()]

In [9]:
driver.data_ingestion(records)

In [10]:
hotSpot = driver.hot_spot_query()

In [11]:
hotSpot.head(30)

,Station,Pollutant,Bad_Readings,Avg_Value,Worst_Value
0,Gwangjin-gu,PM2.5,75,79.32,138.0
1,Yongsan-gu,PM2.5,75,75.75,137.0
2,Mapo-gu,PM2.5,74,73.04,128.0
3,Jungnang-gu,PM2.5,72,77.68,146.0
4,Jongno-gu,PM2.5,72,71.39,121.0
5,Gangbuk-gu,PM2.5,71,73.52,120.0
6,Nowon-gu,PM2.5,71,73.13,112.0
7,Gangdong-gu,PM2.5,71,69.46,133.0
8,Dongjak-gu,PM2.5,70,66.20,237.0
9,Seongdong-gu,PM2.5,70,65.61,126.0


In [12]:
dailyAvg = driver.daily_avg_query()

In [13]:
dailyAvg

,Day,Station,Pollutant,Daily_Avg_Value
0,2017-01-01,Dobong-gu,O3,0.00
1,2017-01-01,Dobong-gu,NO2,0.05
2,2017-01-01,Dobong-gu,CO,0.91
3,2017-01-01,Dobong-gu,SO2,0.01
4,2017-01-01,Dobong-gu,PM10,78.30
...,...,...,...,...
595,2017-01-04,Yongsan-gu,NO2,0.04
596,2017-01-04,Yongsan-gu,CO,0.93
597,2017-01-04,Yongsan-gu,SO2,0.01
598,2017-01-04,Yongsan-gu,PM10,49.42


In [14]:
spark.stop()